# 02 — Classical + XLNet baselines (Wang et al. Table 3 configs)

In [1]:
%cd /content
!rm -rf /content/repo
!git clone https://github.com/kvarun314/pes-mtech-project.git /content/repo
%cd /content/repo
%pip install -q -e ".[colab]"
import sys
sys.path.insert(0, "/content/repo/src")  # editable-install .pth is only read at
# interpreter startup, not picked up by an already-running kernel -- insert the
# path directly so the import below works without a runtime restart.
import agentic_sentiment
print("agentic_sentiment importable:", agentic_sentiment.__file__)

/content
Cloning into '/content/repo'...
remote: Enumerating objects: 710, done.
remote: Counting objects: 100% (710/710), done.
remote: Compressing objects: 100% (354/354), done.
remote: Total 710 (delta 386), reused 624 (delta 308), pack-reused 0 (from 0)
Receiving objects: 100% (710/710), 7.90 MiB | 50.54 MiB/s, done.
Resolving deltas: 100% (386/386), done.
/content/repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.4/71.4 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.8/406.8 kB 61.8 MB/s eta 0:00:00
  Building editable for agentic-sentiment (pyproject.toml) ... done
agentic_sentiment importable: /content/repo/src/agentic_sentiment

In [2]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_RUNS_DIR = "/content/drive/MyDrive/pes-mtech-project/runs"

Mounted at /content/drive


## Load a fresh split of the Phase 1 Kaggle dataset
This is a standalone Colab runtime — `/content/` from notebook 01 does not
carry over. Get `kaggle.json` from https://www.kaggle.com/settings -> Create
New Token, upload it via the sidebar (lands in `/content/`), then run the next
two cells to fetch the dataset into this runtime. This is a fresh random split
of the same CSV the LoRA run trained on, not the identical train/test rows
(Phase 1's own split lives only inside that notebook's run) — the baselines
are compared against each other on this split, not against Phase 1's exact
held-out set.

In [3]:
import json, os
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": "laddu66", "key": "KGAT_8583846640d8ebd958e02227ad600e87"}, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

In [4]:
import os, shutil

os.makedirs("/root/.kaggle", exist_ok=True)
if os.path.exists("/content/kaggle.json"):
    shutil.move("/content/kaggle.json", "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

%pip install -q kaggle
!kaggle datasets download -d arhamrumi/amazon-product-reviews -p /content/data --unzip

Dataset URL: https://www.kaggle.com/datasets/arhamrumi/amazon-product-reviews
License(s): CC0-1.0
100% 115M/115M [00:00<00:00, 443MB/s] 



In [5]:
import pandas as pd

df = pd.read_csv("/content/data/Reviews.csv").dropna(subset=["Text", "Score"])
# Cap the working set: SVM with a non-linear kernel is roughly quadratic in
# row count, so the full ~568k-row CSV is infeasible on a Colab GPU/CPU within
# a reasonable session. 20,000 matches the XLNet cell below for a fair,
# comparable-scale baseline table.
df = df.sample(n=min(len(df), 20000), random_state=42)
texts, labels = df["Text"].tolist(), df["Score"].astype(int).tolist()

In [6]:
from sklearn.model_selection import train_test_split
from agentic_sentiment.baselines.classical import (
    train_decision_tree, train_svm, train_naive_bayes, evaluate,
)

X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)
results = {}
for name, trainer in [("decision_tree", train_decision_tree), ("svm_sigmoid", train_svm),
                      ("naive_bayes", train_naive_bayes)]:
    pipeline = trainer(X_train, y_train)
    results[name] = evaluate(pipeline, X_test, y_test)
    print(name, results[name])

import json, os
os.makedirs("/content/drive/MyDrive/pes-mtech-project", exist_ok=True)
with open("/content/drive/MyDrive/pes-mtech-project/results_run2_baselines.json", "w") as f:
    json.dump(results, f, indent=2)
# Persisted now so a later XLNet disconnect doesn't lose these (cheap to recompute, but why risk it)

decision_tree {'accuracy': 0.584, 'precision': 0.3480727555823835, 'recall': 0.33831132622557064, 'f1': 0.34246354742259993}
svm_sigmoid {'accuracy': 0.697, 'precision': 0.5030422975353456, 'recall': 0.37921732866088426, 'f1': 0.39942131122597935}
naive_bayes {'accuracy': 0.6625, 'precision': 0.4635039072974562, 'recall': 0.43423882925209895, 'f1': 0.44694774657968195}


In [7]:
from agentic_sentiment.train.run_dir import RunDir
from agentic_sentiment.baselines.xlnet import run_training as run_xlnet

run_dir = RunDir(base_dir=DRIVE_RUNS_DIR, run_id="xlnet")
# local_output_dir stays off Drive -- the Drive-side checkpoints/ copy
# (pruned to the 2 most recent) is already durable; staging HF's own
# unpruned checkpoint-* dirs there too would double Drive usage for
# nothing. A reconnect under this same run_id resumes automatically
# from the latest Drive-side checkpoint instead of restarting at step 0.
# Evaluate on the SAME X_test/y_test the classical baselines above used --
# without this, XLNet gets scored on its own internal 80/20 split of
# X_train, which isn't comparable to the classical baselines' numbers.
results["xlnet"] = run_xlnet(
    X_train, y_train, run_dir,
    eval_texts=X_test, eval_labels=y_test, local_output_dir="/content/xlnet_output",
)
print(results["xlnet"])

config.json:   0%|          | 0.00/761 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.44GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

[transformers] The best checkpoint /content/xlnet_output/checkpoint-12000 recorded in the resumed state does not exist anymore. Ignoring it for this run.


Step,Training Loss,Validation Loss


Training Loss,Validation Loss,Step,Accuracy,Precision,Recall,F1
0.990547,1.006888,12000,0.644000,0.253898,0.279534,0.246512


{'eval_loss': 1.0068880319595337, 'eval_accuracy': 0.644, 'eval_precision': 0.25389810990331, 'eval_recall': 0.2795340413064077, 'eval_f1': 0.24651220228943388}


In [9]:
import json, os
from agentic_sentiment.eval.results_log import format_results_entry
from agentic_sentiment.colab_sync import push_repo_changes

with open("/content/drive/MyDrive/pes-mtech-project/results_run2_baselines.json", "w") as f:
    json.dump(results, f, indent=2)

entry = format_results_entry(
    run_label="Classical + XLNet baselines (Wang et al. Table 3 configs)",
    context={"dataset": "Kaggle arhamrumi/amazon-product-reviews", "n_samples": len(texts)},
    metrics_by_name=results,
)
history_row = run_dir.history_row("xlnet")

# Use a fine-grained PAT scoped to just this repo (Contents: read/write).
# A missing GITHUB_TOKEN is fine -- this step is optional documentation.
try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = os.environ.get("GITHUB_TOKEN", "")

if not github_token:
    print("GITHUB_TOKEN not set -- skipping auto-push. RESULTS.md entry:")
    print(entry)
else:
    results_path = "/content/repo/docs/RESULTS.md"
    existing_results = open(results_path).read() if os.path.exists(results_path) else ""
    if entry not in existing_results:
        with open(results_path, "a") as f:
            f.write(entry)
    history_path = "/content/repo/docs/CHECKPOINT_HISTORY.md"
    existing_history = open(history_path).read() if os.path.exists(history_path) else ""
    if history_row not in existing_history:
        with open(history_path, "a") as f:
            f.write(history_row + "\n")
    print(push_repo_changes(
        repo_dir="/content/repo",
        paths=["docs/RESULTS.md", "docs/CHECKPOINT_HISTORY.md"],
        message="Classical + XLNet baseline results", token=github_token,
    ))

Pushed 2 path(s) to main: Classical + XLNet baseline results
